# 🛡️ Análise de Segurança Cibernética e Ataques Digitais
## Projeto G2 — Tema 26
---

## 1. Introdução

Este notebook apresenta uma análise exploratória de dados (EDA) sobre incidentes de segurança cibernética no Brasil entre 2015 e 2024. O objetivo é identificar padrões de ameaças, avaliar impactos financeiros e extrair insights acionáveis para apoio à tomada de decisão em segurança da informação.

## 2. Contextualização da Segurança Cibernética

A segurança cibernética tornou-se um dos pilares estratégicos das organizações modernas. Com a aceleração da transformação digital — especialmente pós-pandemia — o volume e a sofisticação dos ataques cresceram de forma exponencial.

**Principais tipos de ameaças analisadas:**
- **Phishing:** Engenharia social para roubo de credenciais
- **Ransomware:** Sequestro de dados com exigência de resgate
- **Malware:** Software malicioso com diversos objetivos
- **DDoS:** Sobrecarga de servidores para interrupção de serviços
- **Vazamento:** Exposição não autorizada de dados sensíveis

**Impactos típicos:**
- Prejuízo financeiro direto e reputacional
- Interrupção de serviços críticos
- Violação de privacidade e conformidade regulatória (LGPD)

## 3. Explicação da Base de Dados

O dataset `simulacao_ciberseguranca_brasil.csv` é um conjunto de dados simulado contendo **4.440 registros** de incidentes de segurança cibernética.

| Coluna | Tipo | Descrição |
|--------|------|----------|
| ano | int | Ano do incidente (2015–2024) |
| mes | int | Mês do incidente (1–12) |
| data | date | Data completa |
| regiao | str | Região do Brasil |
| uf | str | Estado (sigla) |
| setor | str | Setor organizacional |
| tipo_ataque | str | Categoria do ataque |
| vulnerabilidade | str | Vulnerabilidade explorada |
| incidentes | int | Quantidade de incidentes |
| impacto_financeiro | float | Prejuízo em R$ |
| tempo_recuperacao | float | Horas para recuperação |
| sistemas_afetados | int | Sistemas comprometidos |
| nivel_criticidade | str | Baixo/Médio/Alto/Crítico |
| status_resposta | str | Status da resposta |

In [ ]:
# ── 4. Importações e Carregamento dos Dados ──────────────────────────────────
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sqlalchemy import create_engine
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
pd.set_option('display.max_columns', 20)
print('✅ Bibliotecas carregadas com sucesso')

In [ ]:
# Leitura do CSV
df = pd.read_csv('../dados/simulacao_ciberseguranca_brasil.csv', parse_dates=['data'])
print(f'Shape: {df.shape}')
df.head(10)

## 5. Limpeza e Preparação dos Dados

In [ ]:
# Verificação de valores nulos
print('=== Valores Nulos ===')
print(df.isnull().sum())
print(f'\nTotal de nulos: {df.isnull().sum().sum()}')

# Verificação de duplicatas
print(f'\nDuplicatas: {df.duplicated().sum()}')

# Tipos de dados
print('\n=== Tipos de Dados ===')
print(df.dtypes)

In [ ]:
# Estatísticas descritivas das colunas numéricas
df.describe().round(2)

In [ ]:
# Verificação de categorias únicas
cols_cat = ['regiao', 'setor', 'tipo_ataque', 'vulnerabilidade', 'nivel_criticidade', 'status_resposta']
for col in cols_cat:
    print(f'{col}: {sorted(df[col].unique())}')

## 6. Engenharia de Atributos

In [ ]:
# Criação de novas variáveis derivadas
df['trimestre'] = df['mes'].apply(lambda m: f'T{(m-1)//3 + 1}')
df['impacto_por_incidente'] = df['impacto_financeiro'] / df['incidentes']
df['sistemas_por_incidente'] = df['sistemas_afetados'] / df['incidentes']

# Ordenação de criticidade
ordem_crit = {'Baixo': 1, 'Médio': 2, 'Alto': 3, 'Crítico': 4}
df['crit_ordem'] = df['nivel_criticidade'].map(ordem_crit)

print('Novas colunas criadas:')
print(df[['trimestre', 'impacto_por_incidente', 'sistemas_por_incidente', 'crit_ordem']].head())

In [ ]:
# Persistência em SQLite via SQLAlchemy
import os
os.makedirs('../database', exist_ok=True)
engine = create_engine('sqlite:///../database/ciberseguranca.db')
df.to_sql('incidentes', engine, if_exists='replace', index=False)
print('✅ Dados persistidos no SQLite: database/ciberseguranca.db')

# Exemplo de consulta SQL
with engine.connect() as conn:
    from sqlalchemy import text
    result = conn.execute(text('SELECT setor, SUM(incidentes) as total FROM incidentes GROUP BY setor ORDER BY total DESC'))
    print('\nIncidentes por setor (SQL):')
    for row in result:
        print(f'  {row[0]}: {row[1]:,}')

## 7. KPIs (Indicadores-Chave de Desempenho)

In [ ]:
total_incidentes   = df['incidentes'].sum()
impacto_total      = df['impacto_financeiro'].sum()
tempo_medio        = df['tempo_recuperacao'].mean()
ataque_principal   = df.groupby('tipo_ataque')['incidentes'].sum().idxmax()
setor_principal    = df.groupby('setor')['incidentes'].sum().idxmax()
regiao_critica     = df.groupby('regiao')['incidentes'].sum().idxmax()
vuln_principal     = df.groupby('vulnerabilidade')['incidentes'].sum().idxmax()
pct_critico        = df[df['nivel_criticidade']=='Crítico']['incidentes'].sum() / total_incidentes * 100

print('=' * 50)
print('        📊  KPIs — SEGURANÇA CIBERNÉTICA')
print('=' * 50)
print(f'  Total de Incidentes   : {total_incidentes:>12,}')
print(f'  Impacto Financeiro    : R$ {impacto_total/1e6:>10.2f}M')
print(f'  Tempo Médio Recuper.  : {tempo_medio:>11.1f}h')
print(f'  Ataque Predominante   : {ataque_principal:>12}')
print(f'  Setor Mais Afetado    : {setor_principal:>12}')
print(f'  Região Crítica        : {regiao_critica:>12}')
print(f'  Principal Vuln.       : {vuln_principal:>12}')
print(f'  % Incidentes Críticos : {pct_critico:>11.1f}%')
print('=' * 50)

## 8. Visualizações

In [ ]:
# 8.1 Evolução Temporal dos Incidentes
df_tempo = df.groupby(['ano', 'tipo_ataque'])['incidentes'].sum().reset_index()

fig = px.line(df_tempo, x='ano', y='incidentes', color='tipo_ataque',
              markers=True, title='Evolução Temporal dos Incidentes por Tipo de Ataque',
              labels={'ano':'Ano','incidentes':'Incidentes','tipo_ataque':'Tipo de Ataque'},
              template='plotly_dark')
fig.update_layout(xaxis=dict(dtick=1), width=900, height=450)
fig.show()

In [ ]:
# 8.2 Barras: Frequência por tipo de ataque e por setor
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Incidentes por Tipo de Ataque', 'Incidentes por Setor'))

df_atk = df.groupby('tipo_ataque')['incidentes'].sum().sort_values(ascending=False).reset_index()
df_set = df.groupby('setor')['incidentes'].sum().sort_values(ascending=False).reset_index()

fig.add_trace(go.Bar(x=df_atk['tipo_ataque'], y=df_atk['incidentes'],
                     marker_color='#3b82f6', name='Tipo'), row=1, col=1)
fig.add_trace(go.Bar(x=df_set['setor'], y=df_set['incidentes'],
                     marker_color='#f97316', name='Setor'), row=1, col=2)

fig.update_layout(template='plotly_dark', width=900, height=420,
                  title='Frequência de Incidentes por Categoria', showlegend=False)
fig.show()

In [ ]:
# 8.3 Heatmap mensal
nomes_mes = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']
df_heat = df.groupby(['ano','mes'])['incidentes'].sum().reset_index()
df_pivot = df_heat.pivot(index='mes', columns='ano', values='incidentes').fillna(0)
df_pivot.index = [nomes_mes[i-1] for i in df_pivot.index]

fig = go.Figure(data=go.Heatmap(
    z=df_pivot.values, x=[str(c) for c in df_pivot.columns],
    y=df_pivot.index, colorscale='YlOrRd',
    text=df_pivot.values.astype(int), texttemplate='%{text}'
))
fig.update_layout(template='plotly_dark', title='Heatmap Mensal de Incidentes (Mês × Ano)',
                  xaxis_title='Ano', yaxis_title='Mês', width=900, height=420)
fig.show()

In [ ]:
# 8.4 Dispersão: Impacto Financeiro × Incidentes
df_disp = df.groupby(['setor','tipo_ataque']).agg(
    incidentes=('incidentes','sum'),
    impacto=('impacto_financeiro','sum'),
    sistemas=('sistemas_afetados','mean')
).reset_index()

fig = px.scatter(df_disp, x='incidentes', y='impacto', color='setor',
                 size='sistemas', hover_name='tipo_ataque',
                 template='plotly_dark',
                 title='Correlação: Incidentes × Impacto Financeiro',
                 labels={'incidentes':'Total Incidentes','impacto':'Impacto (R$)'})
fig.update_layout(width=900, height=450)
fig.show()

In [ ]:
# 8.5 Ranking de vulnerabilidades
df_vuln = df.groupby('vulnerabilidade')['incidentes'].sum().sort_values(ascending=False).reset_index()

fig = px.bar(df_vuln, x='vulnerabilidade', y='incidentes',
             color='incidentes', color_continuous_scale='Blues',
             template='plotly_dark', title='Ranking de Vulnerabilidades',
             labels={'vulnerabilidade':'Vulnerabilidade','incidentes':'Total Incidentes'})
fig.update_layout(coloraxis_showscale=False, width=800, height=400)
fig.show()

In [ ]:
# 8.6 Impacto financeiro por setor e região
df_imp = df.groupby(['regiao','setor'])['impacto_financeiro'].sum().reset_index()

fig = px.bar(df_imp, x='regiao', y='impacto_financeiro', color='setor',
             barmode='group', template='plotly_dark',
             title='Impacto Financeiro por Região e Setor',
             labels={'regiao':'Região','impacto_financeiro':'Impacto (R$)'})
fig.update_layout(width=900, height=450)
fig.show()

In [ ]:
# 8.7 Tempo de recuperação por criticidade
ordem = ['Baixo','Médio','Alto','Crítico']
df_rec = df.groupby('nivel_criticidade')['tempo_recuperacao'].mean().reindex(ordem).reset_index()

fig = px.bar(df_rec, x='nivel_criticidade', y='tempo_recuperacao',
             color='nivel_criticidade',
             color_discrete_map={'Baixo':'#22c55e','Médio':'#eab308','Alto':'#f97316','Crítico':'#ef4444'},
             template='plotly_dark', title='Tempo Médio de Recuperação por Criticidade',
             labels={'nivel_criticidade':'Criticidade','tempo_recuperacao':'Horas'})
fig.update_layout(showlegend=False, width=700, height=400)
fig.show()

## 9. Interpretação dos Resultados

### 9.1 Evolução Temporal
Os dados revelam **crescimento consistente** dos ataques ao longo do período 2015–2024, refletindo a expansão da superfície de ataque com a digitalização das organizações brasileiras.

### 9.2 Tipos de Ataque
**Phishing** é o ataque mais frequente, aproveitando-se da vulnerabilidade humana. Ransomware, apesar de menor frequência, apresenta o **maior impacto financeiro por incidente**.

### 9.3 Vulnerabilidades
**Senha fraca** é a vulnerabilidade mais explorada, indicando que políticas básicas de autenticação (uso de senhas fortes, MFA) ainda não estão consolidadas na maioria das organizações brasileiras.

### 9.4 Setores
O setor de **Governo** lidera em volume de incidentes, enquanto o setor **Financeiro** concentra os maiores prejuízos absolutos — um alvo prioritário para ataques sofisticados como ransomware.

### 9.5 Correlação Impacto × Incidentes
Há correlação positiva moderada entre volume de incidentes e impacto financeiro, mas com alta variância — alguns setores sofrem menos ataques com prejuízos desproporcionalmente maiores.

## 10. Conclusão

### 📊 Sumário Executivo

A análise dos **4.440 registros** de incidentes de segurança cibernética no Brasil (2015–2024) revela um cenário de **crescimento contínuo e diversificação das ameaças digitais**.

#### Principais Achados:

| Indicador | Resultado |
|-----------|----------|
| Total de Incidentes | 53.644 |
| Impacto Financeiro Total | R$ 4,4 bilhões |
| Tempo Médio de Recuperação | ~61 horas |
| Ataque Predominante | Phishing |
| Setor Mais Afetado | Governo |
| Principal Vulnerabilidade | Senha fraca |

#### Recomendações Estratégicas:
1. **Autenticação multifator (MFA)** deve ser política obrigatória em todos os setores
2. **Treinamento continuado** em segurança da informação para reduzir falha humana
3. **Planos de resposta a incidentes** testados e documentados, especialmente para ransomware
4. **Monitoramento contínuo** com foco em períodos históricos de maior risco
5. **Compartilhamento de inteligência** entre setores para antecipar ameaças emergentes

> ⚠️ *Este projeto utilizou dados simulados para fins acadêmicos. Conclusões refletem padrões analíticos do dataset gerado.*